## Explore and familiarize with PyTorch

Following: [https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html] 

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

Check CUDA support

In [2]:
# Check if CUDA (GPU) is available
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

CUDA Available: True


In [3]:
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Number of GPUs: 1
GPU 0: NVIDIA GeForce RTX 3060


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Default device for computation: {device}")

Default device for computation: cuda


Import data

In [5]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root=r"..\data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root=r"..\data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [6]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [7]:
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Create neural network

In [8]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            #print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [11]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [12]:
model_1 = NeuralNetwork().to(device)
print(model_1)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_1.parameters(), lr=1e-3)

In [15]:
epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_1, loss_fn, optimizer)
    test(test_dataloader, model_1, loss_fn)
print("Done!")

Epoch 1
-------------------------------
Test Error: 
 Accuracy: 32.8%, Avg loss: 2.174257 

Epoch 2
-------------------------------
Test Error: 
 Accuracy: 55.6%, Avg loss: 1.942794 

Epoch 3
-------------------------------
Test Error: 
 Accuracy: 61.5%, Avg loss: 1.583971 

Epoch 4
-------------------------------
Test Error: 
 Accuracy: 62.5%, Avg loss: 1.295787 

Epoch 5
-------------------------------
Test Error: 
 Accuracy: 63.6%, Avg loss: 1.120198 

Epoch 6
-------------------------------
Test Error: 
 Accuracy: 64.7%, Avg loss: 1.009288 

Epoch 7
-------------------------------
Test Error: 
 Accuracy: 66.2%, Avg loss: 0.933752 

Epoch 8
-------------------------------
Test Error: 
 Accuracy: 67.2%, Avg loss: 0.878998 

Epoch 9
-------------------------------
Test Error: 
 Accuracy: 68.5%, Avg loss: 0.837259 

Epoch 10
-------------------------------
Test Error: 
 Accuracy: 69.9%, Avg loss: 0.804096 

Epoch 11
-------------------------------
Test Error: 
 Accuracy: 71.0%, Avg los

In [16]:
model_2 = NeuralNetwork().to(device)
print(model_2)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_2.parameters(), lr=1e-3)

In [18]:
epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_2, loss_fn, optimizer)
    test(test_dataloader, model_2, loss_fn)
print("Done!")

Epoch 1
-------------------------------
Test Error: 
 Accuracy: 84.4%, Avg loss: 0.424020 

Epoch 2
-------------------------------
Test Error: 
 Accuracy: 85.3%, Avg loss: 0.393017 

Epoch 3
-------------------------------
Test Error: 
 Accuracy: 86.1%, Avg loss: 0.379945 

Epoch 4
-------------------------------
Test Error: 
 Accuracy: 86.8%, Avg loss: 0.358883 

Epoch 5
-------------------------------
Test Error: 
 Accuracy: 87.2%, Avg loss: 0.350930 

Epoch 6
-------------------------------
Test Error: 
 Accuracy: 87.4%, Avg loss: 0.347011 

Epoch 7
-------------------------------
Test Error: 
 Accuracy: 87.9%, Avg loss: 0.336616 

Epoch 8
-------------------------------
Test Error: 
 Accuracy: 87.7%, Avg loss: 0.344076 

Epoch 9
-------------------------------
Test Error: 
 Accuracy: 88.2%, Avg loss: 0.336500 

Epoch 10
-------------------------------
Test Error: 
 Accuracy: 88.1%, Avg loss: 0.366006 

Epoch 11
-------------------------------
Test Error: 
 Accuracy: 87.8%, Avg los